# Part 5: robustness checks on rel-trial (section F)

**Just do Runtime → Run all** (L4 GPU, High-RAM). About **1 hour**.

1. **Deletion test**: retrain LightGBM and the GNN *with* label history, then score them with history masked at inference (all of it, and one linked entity at a time: sponsor, facility, condition, ...).
2. **Blend**: LightGBM + GNN, weight picked on validation. Does the GNN add anything?
3. **Accuracy vs. cost** figure, tables and paired bootstrap for everything.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
OUT = f"{WORK}/results_part5"
CACHE = "/content/cache"  # on the Colab disk, not Drive
os.makedirs(OUT, exist_ok=True)
COMMON = f"--seeds 0 1 2 --cache_dir {CACHE} --results {OUT}"

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (1–2 minutes). **Expect 30 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

Start from the finished rel-trial runs saved in the repo (they are re-used for the blend, the figure and the tests).

In [ ]:
for kind in ("runs", "preds"):
    shutil.copytree(f"results/final/{kind}", f"{OUT}/{kind}", dirs_exist_ok=True)
print(sorted(os.listdir(f"{OUT}/runs/study-outcome")))

## 1. Deletion test: LightGBM (a few minutes)

In [ ]:
!python scripts/run.py lgbm --features agg --mask_eval {COMMON}

## 2. Deletion test: GNN with text + label history (about 45 minutes; the first step embeds the text)

In [ ]:
!python scripts/run.py rdl --text glove --history --epochs 30 --mask_eval {COMMON}

## 3. Blend, tables, significance, figure, download

In [ ]:
from IPython.display import Image, Markdown, display
!python scripts/run.py ensemble {COMMON}
!python scripts/run.py tables --results {OUT}
!python scripts/run.py bootstrap {COMMON}
!python scripts/run.py figures --results {OUT}
!python scripts/run.py explore {COMMON}
for f in ("main.md", "masking.md", "bootstrap.md"):
    display(Markdown(open(f"{OUT}/tables/{f}").read()))
display(Image(f"{OUT}/figures/pareto.png"))

In [ ]:
os.makedirs("/content/part5", exist_ok=True)
for d in ("tables", "figures", "explore", "masking", "runs", "preds", "importance"):
    if os.path.exists(f"{OUT}/{d}"):
        shutil.copytree(f"{OUT}/{d}", f"/content/part5/{d}", dirs_exist_ok=True)
shutil.make_archive("/content/final_results_part5", "zip", "/content/part5", ".")
from google.colab import files
files.download("/content/final_results_part5.zip")